In [36]:
import pandas as pd

In [43]:
from sklearn.preprocessing import LabelEncoder
class EamcetPreprocessor:
  def __init__(self):
    self.df = pd.read_csv(
        'https://raw.githubusercontent.com/saivivekreddydevaram/EAMCET_ALLOTMENT_PREDICTOR/refs/heads/main/EMCET.csv'
      )

    self.metadata = [
        'INST_CODE', 'INST_NAME', 'PLACE', 'DIST', 'COED', 'TYPE',
            'BRANCH', 'BRANCH_NAME', 'YEAR', 'TUITION_FEE', 'AFFILIATED'
      ]
    self.category_cols = [
    'OCB', 'OCG', 'BCAB', 'BCAG', 'BCBB', 'BCBG', 'BCCB', 'BCCG',
    'BCDB', 'BCDG', 'BCEB', 'BCEG', 'SCB', 'SCG', 'STB', 'STG', 'EWSB', 'EWSG'
      ]
    self.encoders = {}


  def transform(self):
    id_cols = [c for c in self.metadata if c in self.df.columns]
    category = [c for c in self.category_cols if c in self.df.columns]
    df_long = pd.melt(
    self.df,
    id_vars = id_cols,
    value_vars = category,
    var_name='CATEGORY_CODE',
    value_name='CUTOFF_RANK'
    )
    df_long['TUITION_FEE'] = pd.to_numeric(df_long['TUITION_FEE'], errors='coerce')
    df_long['TUITION_FEE'] = df_long['TUITION_FEE'].fillna(df_long['TUITION_FEE'].median())
    df_long['CUTOFF_RANK'] = pd.to_numeric(df_long['CUTOFF_RANK'], errors='coerce')
    df_long = df_long.dropna(subset=['CUTOFF_RANK'])
    cat_to_encode = ['INST_CODE', 'PLACE', 'DIST', 'COED', 'TYPE',
                          'BRANCH', 'AFFILIATED', 'GENDER', 'CASTE_CATEGORY']
    for col in cat_to_encode:
            if col in df_long.columns:
                le = LabelEncoder()
                df_long[col] = le.fit_transform(df_long[col].astype(str))
                self.encoders[col] = le
    model_cols = [c for c in cat_to_encode if c in df_long.columns] + ['TUITION_FEE', 'YEAR', 'CUTOFF_RANK']
    model_cols = [c for c in model_cols if c in df_long.columns]
    self.ml_df = df_long[model_cols].dropna()
    return self.ml_df
  def encode_input(self, new_input):
      row = new_input.copy()
      for col, le in self.encoders.items():
          if col in row:
              row[col] = le.transform([str(row[col])])[0]
      return row

preprocessor = EamcetPreprocessor()
ml_ready_df = preprocessor.transform()
print(ml_ready_df.head())

   INST_CODE  PLACE  DIST  COED  TYPE  BRANCH  AFFILIATED  TUITION_FEE  YEAR  \
0          0      6     2     0     1      20          12      48000.0  2019   
1          0      6     2     0     1      30          12      48000.0  2019   
2          0      6     2     0     1      33          12      48000.0  2019   
3          0      6     2     0     1      46          12      48000.0  2019   
4          1     22    11     0     1      13          12      82000.0  2019   

   CUTOFF_RANK  
0      25594.0  
1      51982.0  
2      99633.0  
3      29410.0  
4      41557.0  


In [44]:
feature_cols = [c for c in ['INST_CODE', 'PLACE', 'DIST', 'COED', 'TYPE',
                             'BRANCH', 'AFFILIATED', 'GENDER', 'CASTE_CATEGORY',
                             'TUITION_FEE', 'YEAR']
                if c in ml_ready_df.columns]
X = ml_ready_df[feature_cols]
y = ml_ready_df['CUTOFF_RANK']

In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.dtypes)

INST_CODE        int64
PLACE            int64
DIST             int64
COED             int64
TYPE             int64
BRANCH           int64
AFFILIATED       int64
TUITION_FEE    float64
YEAR             int64
dtype: object


In [46]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

RandomForestRegressor(min_samples_leaf=2, n_estimators=300, n_jobs=-1,
                      random_state=42)

In [47]:
def predict_cutoff(new_input, preprocessor, rf, feature_cols):
    encoded_row = preprocessor.encode_input(new_input)
    input_df = pd.DataFrame([encoded_row])[feature_cols]
    predicted_rank = rf.predict(input_df)
    return predicted_rank[0]

In [48]:
new_input = {
    'INST_CODE': 'ACEG',
    'PLACE': 'GHATKESAR',
    'DIST': 'MDL',
    'COED': 'COED',
    'TYPE': 'PVT',
    'BRANCH': 'CSE',
    'AFFILIATED': 'JNTUH',
    'GENDER': 'Boys',
    'CASTE_CATEGORY': 'OC',
    'TUITION_FEE': 82000,
    'YEAR': 2019
}

predicted_rank = predict_cutoff(new_input,preprocessor, rf, feature_cols)
print(f"Predicted cutoff rank: {predicted_rank:.0f}")

Predicted cutoff rank: 68806


In [49]:
from sklearn.metrics import mean_absolute_error

y_pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.0f}")

MAE: 17323
